--------
Connect Minio
--------
--------

In [4]:
from minio import Minio
from dotenv import load_dotenv
import os

# Load .env
from dotenv import load_dotenv
import os

load_dotenv(".env.local", override=True)

# Read env variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")
bucket_name = os.getenv("MINIO_BUCKET")

# Connect to MinIO
client = Minio(
    endpoint,
    access_key=access_key,
    secret_key=secret_key,
    secure=False
)

TypeError: can only concatenate str (not "NoneType") to str

In [3]:
for key in ["MINIO_ENDPOINT", "MINIO_ACCESS_KEY", "MINIO_SECRET_KEY", "MINIO_BUCKET"]:
    os.environ.pop(key, None)

--------
Create bucket and upload files
--------
--------

In [10]:
# Create bucket if not exists
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)

data_path = "../olist_dataset"

files = [
    "olist_orders_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_order_payments_dataset.csv"
]

for file in files:
    file_path = os.path.join(data_path, file)
    object_name = f"bronze/{file}"

    client.fput_object(
        bucket_name,
        object_name,
        file_path
    )

    print(f"Uploaded {file} to MinIO")

MaxRetryError: HTTPConnectionPool(host='minio', port=9000): Max retries exceeded with url: /olist-data?location= (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x11f9308c0>: Failed to resolve 'minio' ([Errno 8] nodename nor servname provided, or not known)"))

--------
Connect Minio
--------
--------

In [ ]:
from minio import Minio
from pymongo import MongoClient
from dotenv import load_dotenv
import pandas as pd
import os

# Load env
load_dotenv()

# MinIO config
minio_client = Minio(
    os.getenv("MINIO_ENDPOINT"),
    access_key=os.getenv("MINIO_ACCESS_KEY"),
    secret_key=os.getenv("MINIO_SECRET_KEY"),
    secure=False
)

bucket_name = os.getenv("MINIO_BUCKET")

--------
Connect Mongodb
--------
--------

In [ ]:
from dotenv import load_dotenv
import os
from pymongo import MongoClient

load_dotenv()

mongo_uri = os.getenv("MONGO_URI")
mongo_db = os.getenv("MONGO_DB")

client = MongoClient(mongo_uri)
db = client[mongo_db]

--------
Upload files from Minio to Mongodb
--------
--------

In [ ]:
# Files
collections = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "payments": "olist_order_payments_dataset.csv"
}

for name, file in collections.items():
    collection = db[name]

    # Clear old data 
    collection.delete_many({})

    object_name = f"bronze/{file}"
    local_file = f"/tmp/{file}"

    minio_client.fget_object(bucket_name, object_name, local_file)

    df = pd.read_csv(local_file)
    records = df.to_dict(orient="records")

    if records:
        collection.insert_many(records)

    print(f"Loaded {file} into MongoDB")

Loaded olist_orders_dataset.csv into MongoDB
Loaded olist_customers_dataset.csv into MongoDB
Loaded olist_order_items_dataset.csv into MongoDB
Loaded olist_products_dataset.csv into MongoDB
Loaded olist_order_payments_dataset.csv into MongoDB
